# Classical Baseline — TF-IDF + (LogReg | GradientBoosting)

Owner: Jana

Bake-off candidate A (research.md Decision 1). Picks the better of LogisticRegression / GradientBoostingClassifier by macro-F1 on the validation split and saves it to `../artifacts/classifier.joblib`. Prints the SHA-256 of the saved file so it can be pasted into `../artifacts/model_card.md`.


In [ ]:
import hashlib
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline

train = pd.read_csv("data/cleaned/clean_strict_train.csv")
val = pd.read_csv("data/cleaned/clean_strict_val.csv")

candidates = {
    "logreg": Pipeline(
        [
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ]
    ),
    "gb": Pipeline(
        [
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
            ("clf", GradientBoostingClassifier(random_state=42)),
        ]
    ),
}

scores = {}
for name, pipe in candidates.items():
    pipe.fit(train["message"], train["label"])
    pred = pipe.predict(val["message"])
    scores[name] = f1_score(val["label"], pred, average="macro")
    print(name, "macro-F1 val:", scores[name])
    print(classification_report(val["label"], pred, zero_division=0))

best = max(scores, key=scores.get)
print("Picked", best, "with val macro-F1", scores[best])

artifact = Path("../artifacts/classifier.joblib")
joblib.dump(candidates[best], artifact)
print("Wrote", artifact, "sha256:", hashlib.sha256(artifact.read_bytes()).hexdigest())